# Predator-Prey (Lotka-Volterra)

*System Dynamics Studio, model library: Complex behavior*

This is a classic model of population dynamics between two species: a predator (e.g., foxes) and a prey (e.g., rabbits). It illustrates how biological systems create natural oscillations. As the prey population grows, predators have more food and their population increases. However, more predators eventually over-consume the prey, leading to a predator die-off, which then allows the prey population to recover, starting the cycle anew.

**Concepts:** Coupled stocks, Nonlinearity, Oscillation

**Expected behavior:** Endogenous oscillation: predator peaks lag prey peaks.

**Limitations:** Classic Lotka–Volterra: no prey self-limitation, so cycle amplitude depends on initial conditions.

## Model structure

**Stock balances**

$$
\begin{aligned}
\frac{d\,\text{Prey}}{dt} &= \text{prey\_births} - \text{prey\_deaths} \\
\frac{d\,\text{Predators}}{dt} &= \text{predator\_births} - \text{predator\_deaths} \\
\end{aligned}
$$

**Stocks**

| Stock | Initial value | Unit |
|---|---|---|
| `Prey` | 100 | — |
| `Predators` | 20 | — |

**Flows and auxiliaries** (in evaluation order)

$$
\begin{aligned}
\text{prey\_births} &= \text{Prey}\cdot\text{prey\_birth\_rate} \\
\text{prey\_deaths} &= \text{Prey}\cdot\text{Predators}\cdot\text{predation\_rate} \\
\text{predator\_births} &= \text{Prey}\cdot\text{Predators}\cdot\text{predator\_birth\_rate} \\
\text{predator\_deaths} &= \text{Predators}\cdot\text{predator\_death\_rate} \\
\end{aligned}
$$

**Parameters**

| Parameter | Value | Unit |
|---|---|---|
| `prey_birth_rate` | 0.05 | — |
| `predator_birth_rate` | 0.0002 | — |
| `predator_death_rate` | 0.03 | — |
| `predation_rate` | 0.001 | — |

## Setup

NumPy, SciPy and Matplotlib are preinstalled on Google Colab. The step, ramp and pulse helpers
are the time functions available in the app; they are defined here for completeness.

In [ ]:
# Generated by System Dynamics Studio
# Model time: 0 to 500 (time unit not set), output every 0.1
# In-app solver: RK4, fixed dt = 0.1.
# This script integrates with scipy.integrate.odeint (adaptive LSODA), so values
# match the app within numerical tolerance, not digit for digit.
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# Time functions (discontinuous; same definitions as in the app)
def sd_step(height, start, t):
    return np.where(t >= start, height, 0.0)

def sd_ramp(slope, start, *rest):
    *end, t = rest
    until = np.minimum(t, end[0]) if end else t
    return np.where(t > start, slope * (until - start), 0.0)

def sd_pulse(height, start, width, t):
    return np.where((t >= start) & (t < start + width), height, 0.0)

## Parameters and initial values

Edit the values below and rerun the notebook from here to try your own scenarios.

In [ ]:
# 1. Model Parameters (Constants)
params = {
    'prey_birth_rate': 0.05,
    'predator_birth_rate': 0.0002,
    'predator_death_rate': 0.03,
    'predation_rate': 0.001
}

# 2. Initial State (Stocks)
y0 = [
    100,  # Prey
    20  # Predators
]

## Model equations

`model` returns the rate of change of every stock. It is the system of ordinary differential equations shown above.

In [ ]:
# 3. Model Definition
def model(y, t, p):
    # Unpack stocks
    Prey, Predators = y

    # Auxiliaries & Flows (dependency order)
    prey_births = (Prey * p['prey_birth_rate'])
    prey_deaths = ((Prey * Predators) * p['predation_rate'])
    predator_births = ((Prey * Predators) * p['predator_birth_rate'])
    predator_deaths = (Predators * p['predator_death_rate'])

    # Derivatives (Change in Stocks)
    dPrey_dt = prey_births - prey_deaths
    dPredators_dt = predator_births - predator_deaths

    return [dPrey_dt, dPredators_dt]

## Simulation

The model runs from t = 0 to t = 500 time units. The app integrates with RK4 at a fixed
dt = 0.1; this notebook uses SciPy's adaptive `odeint` (LSODA) with tight tolerances, so results agree
with the app within numerical tolerance.

In [ ]:
# 4. Run Simulation
t = np.linspace(0, 500, 5001)
sol = odeint(model, y0, t, args=(params,), rtol=1e-8, atol=1e-10)

In [ ]:
# 5. Post-Processing (Calculate variables for plotting)
Prey = sol[:, 0]
Predators = sol[:, 1]
prey_birth_rate = np.full_like(t, params['prey_birth_rate'])
predator_birth_rate = np.full_like(t, params['predator_birth_rate'])
predator_death_rate = np.full_like(t, params['predator_death_rate'])
predation_rate = np.full_like(t, params['predation_rate'])
prey_births = (Prey * params['prey_birth_rate']) + np.zeros_like(t)
prey_deaths = ((Prey * Predators) * params['predation_rate']) + np.zeros_like(t)
predator_births = ((Prey * Predators) * params['predator_birth_rate']) + np.zeros_like(t)
predator_deaths = (Predators * params['predator_death_rate']) + np.zeros_like(t)

## Results

In [ ]:
# 6. Visualization
plt.figure(figsize=(10, 6))
plt.plot(t, Prey, label='Prey', linewidth=2)
plt.plot(t, Predators, label='Predators', linewidth=2)

plt.title('System Dynamics Simulation (Python)')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Experiment

Starting with 50 % more prey: how do the amplitude and period of the cycle change?

The cell below reruns the model with one change and compares each stock against the baseline.

In [ ]:
STOCKS = ['Prey', 'Predators']

def simulate(param_changes=None, initial_changes=None):
    """Rerun the model with changed parameters and/or initial stock values."""
    p = {**params, **(param_changes or {})}
    y_init = list(y0)
    for stock, value in (initial_changes or {}).items():
        y_init[STOCKS.index(stock)] = value
    return odeint(model, y_init, t, args=(p,), rtol=1e-8, atol=1e-10)

scenario = simulate(param_changes={}, initial_changes={'Prey': 150})

fig, axes = plt.subplots(len(STOCKS), 1, sharex=True, figsize=(10, 2.8 * len(STOCKS)))
axes = np.atleast_1d(axes)
for i, (ax, stock) in enumerate(zip(axes, STOCKS)):
    ax.plot(t, sol[:, i], label='Baseline', linewidth=2)
    ax.plot(t, scenario[:, i], '--', label='Prey(0) × 1.5', linewidth=2)
    ax.set_ylabel(stock)
    ax.grid(True, alpha=0.3)
axes[0].legend(loc='best')
axes[-1].set_xlabel('Time')
fig.suptitle('Baseline vs. scenario')
plt.tight_layout()
plt.show()

## Questions to explore

- Which population peaks first, and why?
- Run with Euler at dt = 0.5 and compare with RK4. What happens to the cycle amplitude?